In [20]:
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
import collections
import itertools
import tqdm
from copy import deepcopy
from functools import lru_cache

In [21]:
np.random.seed(0)

In [22]:
def normalize_priors(priors):
    if np.sum(priors) == 0:
        return np.zeros_like(priors)
    return priors / np.sum(priors)

In [23]:
def best_response_vectorized(X, thresholds, priors, c):
    posteriors = normalize_priors(priors)
    utilities_expected = np.array([
        np.dot(posteriors, thresholds[j] >= thresholds)
        for j in range(len(thresholds))
    ])

    pass_matrix = X[:, None] >= thresholds[None, :]
    utility_stay = pass_matrix @ posteriors

    X_col = X[:, None]
    thresholds_row = thresholds[None, :]

    feasible = thresholds_row > X_col
    utility_jump = utilities_expected[None, :] - c * np.abs(thresholds_row - X_col)
    utility_jump[~feasible] = -np.inf

    utilities = np.concatenate([utility_stay[:, None], utility_jump],axis=1)

    best_idx = np.argmax(utilities, axis=1)
    X_p = np.where(best_idx == 0, X, thresholds[best_idx - 1])

    return X_p

In [24]:
def accuracy_loss_vectorized(X, X_p, thresholds, priors, threshold_true):
    Y_true = (X >= threshold_true).astype(float)
    Y_p = (X_p[:, None] >= thresholds[None, :]).astype(float)
    losses = np.abs(Y_true[:, None] - Y_p).mean(axis=0)
    posteriors = normalize_priors(priors)
    return np.dot(losses, posteriors)

In [25]:
def evaluate_partition(X, partition, thresholds, priors, threshold_true, c, return_all=False):
    thresholds_p = thresholds[partition]
    priors_p = priors[partition]
    X_p = best_response_vectorized(X, thresholds_p, priors_p, c)
    acc_loss_p = accuracy_loss_vectorized(X, X_p, thresholds_p, priors_p, threshold_true)
    if return_all:
        return acc_loss_p, thresholds_p, priors_p
    return acc_loss_p

def evaluate_system(X, partitions, thresholds, priors, threshold_true, c):
    acc_loss = 0.
    for partition in partitions:
        acc_loss_p = evaluate_partition(X, partition, thresholds, priors, threshold_true, c)
        acc_loss += acc_loss_p * np.sum(priors[partition])
    return acc_loss

In [26]:
def set_partitions(collection):
    if len(collection) == 1:
        yield [collection]
        return

    first = collection[0]
    for smaller in set_partitions(collection[1:]):
        for i in range(len(smaller)):
            yield smaller[:i] + [[first] + smaller[i]] + smaller[i+1:]
        yield [[first]] + smaller

In [27]:
def find_partitions_optimal(X, thresholds, priors, threshold_true, c):
    indices = list(range(len(thresholds)))
    partitions_set = list(set_partitions(indices))

    @lru_cache(maxsize=None)
    def evaluate_partition_cached(partition_tuple):
        partition = list(partition_tuple)
        acc_loss_p = evaluate_partition(
            X, partition, thresholds, priors, threshold_true, c
        )
        return acc_loss_p * np.sum(priors[partition])

    best_partition = None
    best_loss = np.inf

    for partitions in partitions_set:
        acc_loss = 0.0
        for partition in partitions:
            partition_tuple = tuple(sorted(partition))
            acc_loss += evaluate_partition_cached(partition_tuple)

        if acc_loss < best_loss:
            best_loss = acc_loss
            best_partition = deepcopy(partitions)

    return best_partition

In [28]:
def display_queue(Q, P):
    res = "[  "
    for (a_id, b_id) in Q:
        res += f"({P[a_id]}, {P[b_id]})  "
    res += "]"
    print(res)

def find_partitions_greedy(X, thresholds, priors, threshold_true, c, display=False):
    P = {}
    next_id = 0
    partitions = [[i] for i in range(len(priors))]
    for block in partitions:
        P[next_id] = list(block)
        next_id += 1


    Q = collections.deque(itertools.combinations(P.keys(), 2))

    while Q:
        if display:
            display_queue(Q, P)
        a_id, b_id = Q.popleft()
        if a_id not in P.keys() or b_id not in P.keys():
            continue

        a = P[a_id]
        b = P[b_id]

        acc_loss_a = evaluate_partition(X, a, thresholds, priors, threshold_true, c)
        acc_loss_b = evaluate_partition(X, b, thresholds, priors, threshold_true, c)
        lhs = acc_loss_a * np.sum(priors[a]) + acc_loss_b * np.sum(priors[b])

        ab = sorted(a + b)
        acc_loss_ab = evaluate_partition(X, ab, thresholds, priors, threshold_true, c)
        rhs = acc_loss_ab * np.sum(priors[ab])

        if lhs - rhs > -1e-6:
            del P[a_id]
            del P[b_id]

            Q = collections.deque(
                (x, y)
                for (x, y) in Q
                if x not in {a_id, b_id} and y not in {a_id, b_id}
            )

            new_id = next_id
            next_id += 1
            P[new_id] = ab

            for c_id in P.keys():
                if c_id != new_id:
                    Q.append((new_id, c_id))

            # Q = collections.deque(sorted(Q, key=lambda x: P[x[0]]))
    return list(P.values())

In [29]:
def approximation_ratio(loss_optimal, loss_greedy):
    if (1 - loss_greedy) == 0:
        return np.nan
    return (1 - loss_optimal) / (1 - loss_greedy)

In [30]:
x_min, x_max, x_disc = 0., 1., 1e-4
X = np.arange(x_min, x_max+x_disc, x_disc).round(4)
N_REPS = 100
N_THRESHOLDS = 6

In [31]:
np.random.seed(0)

def sweep(param_values, n_reps=N_REPS):
    d = {"i": [], "alpha": [], "r": []}
    c = 0.5
    threshold_true = 0.9999
    for i in tqdm.trange(n_reps):
        thresholds = np.sort(np.random.uniform(1e-9, 1, N_THRESHOLDS))
        # threshold_true = np.median(thresholds)
        for val in param_values:   
            priors = np.random.dirichlet([val] * N_THRESHOLDS)
            partition_opt = find_partitions_optimal(X, thresholds, priors, threshold_true, c)
            partition_greedy = find_partitions_greedy(X, thresholds, priors, threshold_true, c)
            if partition_opt is None:
                print(partition_opt)
                print(thresholds)
                print(val)
                print(priors)
            loss_opt = evaluate_system(X, partition_opt, thresholds, priors, threshold_true, c)
            loss_greedy = evaluate_system(X, partition_greedy, thresholds, priors, threshold_true, c)
            r  = approximation_ratio(loss_opt, loss_greedy)
            if not np.isnan(r):
                d["i"].append(i)
                d["alpha"].append(val)
                d["r"].append(r.item())

    return d

In [ ]:
alpha_values = np.arange(9,10.1, 0.1)
d = sweep(alpha_values, n_reps=1000)

100%|██████████| 1000/1000 [03:35<00:00,  4.65it/s]


In [36]:
df = pd.DataFrame(d)
df_group = df.groupby(["alpha"], as_index=False)
r_a, s_a = df_group.mean()["r"].values, df_group.std()["r"].values

In [37]:
fig = go.Figure()
fig.add_scatter(
    x = alpha_values,
    y = np.maximum(r_a - s_a, 1),
    mode = "lines",
    line_color = "rgba(37, 99, 235, 0)",
    showlegend=False
    )

fig.add_scatter(
    x = alpha_values,
    y = r_a + s_a,
    mode = "lines",
    line_color = "rgba(37, 99, 235, 0)",
    fill = "tonexty",
    fillcolor = "rgba(37, 99, 235, 0.2)",
    showlegend=False
    )

fig.add_scatter(
    x = alpha_values,
    y = r_a,
    mode = "lines+markers",
    line_color = "rgba(37, 99, 235, 1)",
    showlegend=False
)

fig.update_layout(
    width=550, height=450,
    font=dict(family="iosevka"),
    xaxis=dict(title="Dirichlet α  (higher = more uniform)", type="log"),
    yaxis=dict(title="Approximation Ratio"),
    margin=dict(t=25,b=25,l=25,r=25)
    )

In [38]:
fig.write_image("../figures/effect_of_prior_concentration_alpha_above_100.pdf")